In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from urllib.parse import unquote_plus
import os

In [2]:
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

VISUAL_EMB_PATH      = "brain/quintuplet_embeddings.npy"
VISUAL_POI_IDS_PATH  = "brain/quintuplet_embeddings_poi_ids.npy"
CTX_EMB_PATH         = "brain/context_embeddings_391d.npy"
CTX_IDS_PATH         = "brain/context_photo_ids.npy"
POI_MAP_PATH         = "brain/photo_poi_mapping.csv"
IDS_DIR              = "brain/ids/"
CSV_PATH             = "brain/usa_europe_geotagged.csv"
NUM_CHUNKS           = 768

OUTPUT_FILTERED_EMB  = "brain/filtered_visual_embeddings.npy"
OUTPUT_FILTERED_IDS  = "brain/filtered_photo_ids.npy"
OUTPUT_FILTERED_POIS = "brain/filtered_poi_ids.npy"

VISUAL_DIM     = 256
CONTEXTUAL_DIM = 391
COMBINED_DIM   = VISUAL_DIM + CONTEXTUAL_DIM

K_PERCENT = 0.3
K_MIN     = 3
K_MAX     = 30

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


In [3]:
print("Loading visual embeddings...")
visual_emb    = np.load(VISUAL_EMB_PATH)

print("Loading contextual embeddings...")
ctx_emb       = np.load(CTX_EMB_PATH).astype(np.float32)
ctx_photo_ids = np.load(CTX_IDS_PATH, allow_pickle=True).astype(int)

print("Loading POI mapping...")
poi_map       = pd.read_csv(POI_MAP_PATH)
poi_map['photoid'] = poi_map['photoid'].astype(int)
poi_map_dict  = poi_map.set_index('photoid')['poi_id'].to_dict()

print("Loading CSV...")
df            = pd.read_csv(CSV_PATH)
df['photoid'] = df['photoid'].astype(int)
df_indexed    = df.set_index('photoid')

print(f"Visual embeddings    : {visual_emb.shape}")
print(f"Contextual embeddings: {ctx_emb.shape}")
print(f"CSV rows             : {len(df)}")

Loading visual embeddings...
Loading contextual embeddings...
Loading POI mapping...
Loading CSV...
Visual embeddings    : (3744663, 256)
Contextual embeddings: (4927028, 391)
CSV rows             : 7672417


In [4]:
print("Replaying NB6 filter to get aligned photo IDs...")
vis_photo_ids = []
for i in tqdm(range(NUM_CHUNKS), desc="Loading ID chunks"):
    ip = os.path.join(IDS_DIR, f"ids_chunk_{i}.npy")
    if not os.path.exists(ip): continue
    ids  = np.load(ip, allow_pickle=True).astype(int)
    pois = np.array([poi_map_dict.get(pid, -1) for pid in ids])
    mask = pois != -1
    vis_photo_ids.extend(ids[mask].tolist())

vis_photo_ids = np.array(vis_photo_ids, dtype=int)
assert len(vis_photo_ids) == len(visual_emb), "MISMATCH"
print(f"✓ Visual IDs matched: {len(vis_photo_ids):,}")

vis_pid_to_idx = {pid: idx for idx, pid in enumerate(vis_photo_ids)}
ctx_pid_to_idx = {pid: idx for idx, pid in enumerate(ctx_photo_ids)}

vis_pid_set  = set(vis_pid_to_idx.keys())
ctx_pid_set  = set(ctx_pid_to_idx.keys())
poi_pid_set  = set(poi_map_dict.keys())
common_pids  = np.array(sorted(vis_pid_set & ctx_pid_set & poi_pid_set), dtype=int)

print(f"Common aligned samples: {len(common_pids):,}")

vis_indices  = np.array([vis_pid_to_idx[pid] for pid in common_pids])
ctx_indices  = np.array([ctx_pid_to_idx[pid] for pid in common_pids])

aligned_vis  = visual_emb[vis_indices]
aligned_ctx  = ctx_emb[ctx_indices]
aligned_poi  = np.array([poi_map_dict[pid] for pid in common_pids])
aligned_pid  = common_pids

print(f"aligned_vis : {aligned_vis.shape}")
print(f"aligned_ctx : {aligned_ctx.shape}")

Replaying NB6 filter to get aligned photo IDs...


Loading ID chunks: 100%|█████████████████████████████████████████████████████████████| 768/768 [00:03<00:00, 203.06it/s]


✓ Visual IDs matched: 3,744,663
Common aligned samples: 3,744,663
aligned_vis : (3744663, 256)
aligned_ctx : (3744663, 391)


In [5]:
class CrossModalProjector(nn.Module):
    def __init__(self, vis_dim=256, ctx_dim=391, shared_dim=128):
        super().__init__()
        self.vis_proj = nn.Sequential(
            nn.Linear(vis_dim, shared_dim),
            nn.BatchNorm1d(shared_dim),
            nn.GELU()
        )
        self.ctx_proj = nn.Sequential(
            nn.Linear(ctx_dim, shared_dim),
            nn.BatchNorm1d(shared_dim),
            nn.GELU()
        )

    def forward(self, vis, ctx):
        v = F.normalize(self.vis_proj(vis), p=2, dim=1)
        c = F.normalize(self.ctx_proj(ctx), p=2, dim=1)
        return (v * c).sum(dim=1)

projector   = CrossModalProjector().to(DEVICE)
projector.eval()

SCORE_BATCH = 8192
raw_scores  = []

with torch.no_grad():
    for i in tqdm(range(0, len(aligned_vis), SCORE_BATCH), desc="Scoring photos"):
        vis_b  = torch.tensor(aligned_vis[i:i+SCORE_BATCH], dtype=torch.float32).to(DEVICE)
        ctx_b  = torch.tensor(aligned_ctx[i:i+SCORE_BATCH], dtype=torch.float32).to(DEVICE)
        scores = projector(vis_b, ctx_b).cpu().numpy()
        raw_scores.append(scores)

raw_scores = np.concatenate(raw_scores, axis=0)
all_scores = (raw_scores - raw_scores.min()) / (raw_scores.max() - raw_scores.min() + 1e-8)

print(f"Score stats — Min: {all_scores.min():.4f} | Max: {all_scores.max():.4f} | Mean: {all_scores.mean():.4f} | Std: {all_scores.std():.4f}")

Scoring photos: 100%|████████████████████████████████████████████████████████████████| 458/458 [00:02<00:00, 164.80it/s]


Score stats — Min: 0.0000 | Max: 1.0000 | Mean: 0.5111 | Std: 0.1327


In [6]:
place_keywords = [
    # Nature & Landscape
    'nature', 'landscape', 'water', 'sky', 'beach', 'river', 'lake',
    'ocean', 'sea', 'mountain', 'mountains', 'forest', 'trees', 'tree',
    'sunset', 'sunrise', 'clouds', 'snow', 'waterfall', 'valley', 'canyon',
    'desert', 'coast', 'island', 'rocks', 'hills', 'field', 'grass',
    'stream', 'creek', 'pond', 'bay', 'harbor', 'harbour', 'shore',
    'scenery', 'scenic', 'outdoors', 'outdoor', 'rural', 'countryside',
    'woods', 'flora', 'cascades', 'alps', 'pacific', 'pacific ocean',
    # Urban & Architecture
    'architecture', 'city', 'urban', 'street', 'building', 'buildings',
    'bridge', 'downtown', 'skyline', 'skyscraper', 'cityscape', 'road',
    'tower', 'wall', 'window', 'roof', 'door', 'stairs', 'sign',
    'construction', 'graffiti', 'street art', 'streetart', 'mural',
    'industrial', 'abandoned', 'decay', 'rust', 'subway', 'metro',
    'station', 'railway', 'railroad', 'rail', 'canal', 'pier', 'port',
    'square', 'panorama', 'reflection', 'reflections', 'interior',
    # Landmarks & Tourism
    'travel', 'museum', 'park', 'church', 'historic', 'history',
    'monument', 'statue', 'sculpture', 'memorial', 'cathedral', 'library',
    'garden', 'gardens', 'fountain', 'palace', 'ruins', 'ancient',
    'medieval', 'roman', 'gothic', 'heritage', 'landmark', 'national park',
    'tourist', 'tourism', 'vacation', 'trip', 'roadtrip', 'road trip',
    'hiking', 'hike', 'trail', 'camp', 'camping', 'visit', 'visitor',
    'nrhp', 'smithsonian', 'unesco', 'national trust', 'zoo', 'aquarium',
    'cemetery', 'fort', 'lighthouse', 'gate', 'exhibit', 'exhibition',
    'gallery', 'university', 'campus', 'national mall', 'castle',
    # Cities
    'london', 'paris', 'new york', 'nyc', 'san francisco', 'seattle',
    'chicago', 'washington dc', 'brooklyn', 'manhattan', 'los angeles',
    'berlin', 'rome', 'amsterdam', 'barcelona', 'madrid', 'vienna',
    'venice', 'prague', 'budapest', 'istanbul', 'edinburgh', 'glasgow',
    'toronto', 'vancouver', 'montreal', 'ottawa', 'dublin', 'brussels',
    'copenhagen', 'stockholm', 'helsinki', 'portland', 'boston',
    'philadelphia', 'miami', 'denver', 'austin', 'nashville',
    'new orleans', 'las vegas', 'hollywood', 'florence', 'yosemite',
    # Transport
    'airport', 'train', 'boat', 'ship', 'ferry', 'plane', 'airplane',
    'aircraft', 'aviation', 'flight', 'sailing', 'cruise', 'transport',
    'bus', 'highway', 'locomotive',
    # Events & Culture
    'festival', 'parade', 'carnival', 'market', 'fair', 'concert',
    'theater', 'theatre', 'art', 'painting', 'vintage', 'antique',
    'war', 'military', 'protest', 'rally', 'celebration',
]

def is_place_photo(pid):
    if int(pid) not in df_indexed.index: return False
    row  = df_indexed.loc[int(pid)]
    text = " ".join([
        unquote_plus(str(row.get('title',       '') or '')),
        unquote_plus(str(row.get('usertags',    '') or '')),
        unquote_plus(str(row.get('description', '') or ''))
    ]).lower()
    return any(kw in text for kw in place_keywords)

print("Place keywords ready.")

Place keywords ready.


In [7]:
class SelfAttentionScorer(nn.Module):
    def __init__(self, input_dim=647, num_heads=4, dropout=0.1):
        super().__init__()
        self.proj       = nn.Linear(input_dim, 256)
        self.attn       = nn.MultiheadAttention(256, num_heads,
                                                dropout=dropout,
                                                batch_first=True)
        self.score_head = nn.Sequential(
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x          = self.proj(x).unsqueeze(0)      # (1, N, 256)
        out, _     = self.attn(x, x, x)             # self-attention
        out        = out.squeeze(0)                  # (N, 256)
        return torch.sigmoid(self.score_head(out).squeeze(-1))  # (N,)

sa_scorer = SelfAttentionScorer(input_dim=COMBINED_DIM).to(DEVICE)
sa_scorer.eval()

poi_to_sample_idx = {}
for idx, poi in enumerate(aligned_poi):
    poi_to_sample_idx.setdefault(poi, []).append(idx)

filtered_vis  = []
filtered_ctx  = []
filtered_pois = []
filtered_pids = []
k_values      = []
removed_junk  = 0

for poi, indices in tqdm(poi_to_sample_idx.items(), desc="Filtering"):
    indices = np.array(indices)

    # Step 1: Hard filter — place photos only (cross-modal score threshold)
    place_mask    = np.array([is_place_photo(aligned_pid[i]) for i in indices])
    place_indices = indices[place_mask]
    removed_junk += (len(indices) - len(place_indices))

    if len(place_indices) < K_MIN:
        continue

    # Step 2: Self-attention scoring within place photos of this POI
    vis_t    = torch.tensor(aligned_vis[place_indices], dtype=torch.float32).to(DEVICE)
    ctx_t    = torch.tensor(aligned_ctx[place_indices], dtype=torch.float32).to(DEVICE)
    combined = torch.cat([vis_t, ctx_t], dim=-1)  # (N, 647)

    with torch.no_grad():
        if len(place_indices) > 1000:
            attn_scores = []
            for j in range(0, len(place_indices), 1000):
                chunk = combined[j:j+1000]
                attn_scores.append(sa_scorer(chunk).cpu().numpy())
            attn_scores = np.concatenate(attn_scores)
        else:
            attn_scores = sa_scorer(combined).cpu().numpy()

    # Step 3: Combine cross-modal score + self-attention score
    cross_scores = all_scores[place_indices]
    final_scores = 0.6 * cross_scores + 0.4 * attn_scores

    # Step 4: Dynamic top-K
    n = len(place_indices)
    k = max(K_MIN, min(K_MAX, int(K_PERCENT * n)))
    k_values.append(k)

    if n <= K_MIN:
        selected = place_indices
    else:
        top_k    = np.argsort(final_scores)[-k:]
        selected = place_indices[top_k]

    filtered_vis.append(aligned_vis[selected])
    filtered_ctx.append(aligned_ctx[selected])
    filtered_pois.extend([poi] * len(selected))
    filtered_pids.extend(aligned_pid[selected].tolist())

filtered_vis  = np.concatenate(filtered_vis,  axis=0)
filtered_ctx  = np.concatenate(filtered_ctx,  axis=0)
filtered_pois = np.array(filtered_pois)
filtered_pids = np.array(filtered_pids)

print(f"Before filtering : {len(aligned_ctx):,}")
print(f"Junk removed     : {removed_junk:,}")
print(f"After filtering  : {len(filtered_vis):,}")
print(f"Reduction        : {100*(1 - len(filtered_vis)/len(aligned_ctx)):.1f}%")
print(f"Avg K per POI    : {np.mean(k_values):.1f}")
print(f"Min K / Max K    : {min(k_values)} / {max(k_values)}")

Filtering: 100%|███████████████████████████████████████████████████████████████| 201623/201623 [27:41<00:00, 121.33it/s]


Before filtering : 3,744,663
Junk removed     : 567,869
After filtering  : 767,210
Reduction        : 79.5%
Avg K per POI    : 4.5
Min K / Max K    : 3 / 30


In [8]:
np.save(OUTPUT_FILTERED_EMB,  filtered_vis)
np.save(OUTPUT_FILTERED_IDS,  filtered_pids)
np.save(OUTPUT_FILTERED_POIS, filtered_pois)

# Save combined (visual + contextual) for Module C input
combined_filtered = np.concatenate([filtered_vis, filtered_ctx], axis=1)
np.save(OUTPUT_FILTERED_EMB.replace(".npy", "_combined.npy"), combined_filtered)

print(f"Filtered visual embeddings : {filtered_vis.shape}")
print(f"Filtered combined embeddings: {combined_filtered.shape}")
print(f"Photo IDs                  : {filtered_pids.shape}")
print(f"POI IDs                    : {filtered_pois.shape}")

Filtered visual embeddings : (767210, 256)
Filtered combined embeddings: (767210, 647)
Photo IDs                  : (767210,)
POI IDs                    : (767210,)
